# **GPT Judge Evaluation**
This notebook measures *Confirmation Bias* using LLM-as-a-judge (with gpt-4o), instructing it to rate the coherence on a scale of 0-10.

In [ ]:
import sys
import os
import pandas as pd

# Add the parent directory to the system path to allow imports from the src directory
sys.path.append(os.path.abspath('..'))

from src.evaluators.gpt_judge import compute_gpt_metrics

## **Data Loading**
Upload one of the logs generated in the pipeline located in the `data/interim/` folder.

In [ ]:
# Define the path to the generated results (JSONL format)
file_path = "../data/interim/3_fever_deepseek_r1_results.jsonl"
# file_path = "../data/interim/4_truthfulqa_deepseek_r1_results.jsonl"
# file_path = "../data/interim/5_mmlu_pro_deepseek_r1_results.jsonl"

try:
    # Load the generated results into a DataFrame
    df_results = pd.read_json(file_path, lines=True)
    print(f"Loaded {len(df_results)} samples.")
    display(df_results.head(1))
except FileNotFoundError:
    print(f"File not found at the requested path ({file_path}).")

## **Evaluation**
Compute the evaluation metrics by sending the generated results to GPT, and extract the final score (parsed list).

In [ ]:
# Compute the GPT metrics
df_evaluated = compute_gpt_metrics(df_results, model="gpt-4o", sleep_time=1.0)

## **Summary**
Compute summary statistics on the evaluated results, and save the final DataFrame to `data/processed/` for later use.

In [ ]:
# Aggregation of the average scores for each prompt type
mean_neutral = df_evaluated["score_neutral"].mean()
mean_leading = df_evaluated["score_leading"].mean()
mean_contra = df_evaluated["score_contradictory"].mean()

print("Results of the GPT Judge Evaluation:")
print(f"Average Support (Neutral Prompt):        {mean_neutral:.2f} / 10")
print(f"Average Support (Leading Prompt):        {mean_leading:.2f} / 10")
print(f"Average Support (Contradictory Prompt):  {mean_contra:.2f} / 10")

# If the Leading prompt increases adherence and the Contradictory prompt decreases it significantly, 
# the numerical discrepancy indicates that the tested model suffers from confirmation bias.

display(df_evaluated[["sample", "model", "claim", "score_neutral", "score_leading", "score_contradictory"]].head(10))